# Purpose

This notebook conducts a small public-data feasibility study for the Regional Entrepreneurship Intelligence project. It tests whether a pilot Metropolitan Statistical Area x 2-digit NAICS x Year panel can be assembled without beginning production ETL, a database, dashboard code, or final modeling.

In [ ]:
import json
import math
import re

import numpy as np
import pandas as pd
import requests
import statsmodels.formula.api as smf

YEARS = list(range(2018, 2024))
SECTORS = ["11", "21", "22", "23", "31-33", "42", "44-45", "48-49", "51", "52", "53", "54", "55", "56", "61", "62", "71", "72", "81"]
BDS_MSA_SECTOR_URL = "https://www2.census.gov/programs-surveys/bds/tables/time-series/2023/bds2023_msa_sec.csv"
QCEW_AREA_TITLES_URL = "https://data.bls.gov/cew/doc/titles/area/area_titles.csv"
PILOT_PATTERNS = {
    "Huntsville, AL": ["huntsville, al"],
    "Nashville, TN": ["nashville", "davidson", "murfreesboro", "franklin"],
    "Chattanooga, TN-GA": ["chattanooga"],
    "Florence-Muscle Shoals, AL": ["florence-muscle shoals", "florence-muscle", "shoals"],
    "Austin, TX": ["austin", "round rock"],
}

def normalize_title(value):
    return re.sub(r"\s+", " ", str(value).lower().replace("-", " ")).strip()

def pct(numerator, denominator):
    return round(100 * numerator / denominator, 1) if denominator else 0.0

def summarize_measure(df, columns, key_cols):
    rows = []
    total_cells = df[key_cols].drop_duplicates().shape[0]
    for col in columns:
        series = pd.to_numeric(df[col], errors="coerce") if col in df else pd.Series(dtype="float")
        rows.append({
            "measure": col,
            "available_rows": int(series.notna().sum()),
            "missing_rows": int(series.isna().sum()),
            "percent_missing": pct(series.isna().sum(), len(df)),
            "zero_rows": int((series == 0).sum()),
            "percent_zero": pct((series == 0).sum(), len(df)),
            "usable_msa_sector_year_cells": int(df.loc[series.notna(), key_cols].drop_duplicates().shape[0]) if col in df else 0,
            "total_candidate_cells": total_cells,
        })
    return pd.DataFrame(rows)

# Pilot MSAs

Pilot MSAs are verified from official BLS QCEW area titles. BLS uses compact MSA area codes such as `C2662`, while BDS uses five-digit CBSA codes such as `26620`.

In [ ]:
area = pd.read_csv(QCEW_AREA_TITLES_URL, dtype=str)
msa = area[area["area_fips"].str.startswith("C", na=False)].copy()
msa["norm"] = msa["area_title"].map(normalize_title)

pilot_rows = []
for label, words in PILOT_PATTERNS.items():
    matches = msa.copy()
    for word in words:
        matches = matches[matches["norm"].str.contains(normalize_title(word), regex=False)]
    row = matches.iloc[0]
    area_digits = re.sub(r"\D", "", row["area_fips"])
    cbsa_code = f"{area_digits}0"
    states_match = re.search(r",\s*([A-Z-]+)\s+MSA", row["area_title"])
    pilot_rows.append({
        "pilot_label": label,
        "cbsa_code": cbsa_code,
        "qcew_area_fips": row["area_fips"],
        "official_name": row["area_title"],
        "states": states_match.group(1) if states_match else None,
    })

pilot = pd.DataFrame(pilot_rows)
pilot

# Data Sources

The feasibility study uses official Census BDS bulk CSV files and BLS QCEW Open Data CSV slices. Census CBP and ACS data queries redirected to a missing-key page in this environment, so they are documented as compatible sources requiring a Census API key for full retrieval.

# BDS Feasibility

In [ ]:
usecols = ["year", "msa", "sector", "firms", "estabs", "emp", "denom", "estabs_entry", "estabs_entry_rate", "job_creation_births", "job_creation_rate_births", "job_creation", "job_creation_rate"]
chunks = []
pilot_codes = set(pilot["cbsa_code"])
for chunk in pd.read_csv(BDS_MSA_SECTOR_URL, usecols=usecols, dtype={"msa": str, "sector": str}, chunksize=200_000):
    chunk["msa"] = chunk["msa"].astype(str).str.zfill(5)
    keep = chunk["msa"].isin(pilot_codes) & chunk["year"].isin(YEARS) & chunk["sector"].isin(SECTORS)
    chunks.append(chunk.loc[keep].copy())

bds = pd.concat(chunks, ignore_index=True).rename(columns={"msa": "cbsa_code", "sector": "naics_sector"})
for col in ["firms", "estabs", "emp", "denom", "estabs_entry", "estabs_entry_rate", "job_creation_births", "job_creation_rate_births", "job_creation", "job_creation_rate"]:
    bds[col] = pd.to_numeric(bds[col], errors="coerce")
bds = bds.merge(pilot[["cbsa_code", "pilot_label", "official_name"]], on="cbsa_code", how="left")
bds_measures = summarize_measure(bds, ["estabs_entry", "estabs_entry_rate", "job_creation_births", "job_creation_rate_births", "firms", "estabs"], ["cbsa_code", "naics_sector", "year"])
bds_measures

# QCEW Feasibility

In [ ]:
qcew_frames = []
for _, area_row in pilot.iterrows():
    for year in YEARS:
        url = f"https://data.bls.gov/cew/data/api/{year}/a/area/{area_row['qcew_area_fips']}.csv"
        df = pd.read_csv(url, dtype={"industry_code": str, "area_fips": str, "own_code": str})
        df = df[(df["own_code"].eq("5")) & (df["industry_code"].isin(SECTORS)) & (df["agglvl_code"].astype(str).eq("44"))].copy()
        df["pilot_label"] = area_row["pilot_label"]
        df["cbsa_code"] = area_row["cbsa_code"]
        df["official_name"] = area_row["official_name"]
        qcew_frames.append(df)

qcew = pd.concat(qcew_frames, ignore_index=True).rename(columns={"industry_code": "naics_sector"})
for col in ["annual_avg_emplvl", "annual_avg_estabs", "total_annual_wages", "avg_annual_pay"]:
    qcew[col] = pd.to_numeric(qcew[col], errors="coerce")
qcew = qcew.sort_values(["cbsa_code", "naics_sector", "year"])
for col, out in [("annual_avg_emplvl", "employment_growth"), ("annual_avg_estabs", "establishment_growth"), ("total_annual_wages", "payroll_growth"), ("avg_annual_pay", "wage_growth")]:
    qcew[out] = qcew.groupby(["cbsa_code", "naics_sector"])[col].pct_change().replace([np.inf, -np.inf], np.nan)
qcew_measures = summarize_measure(qcew, ["annual_avg_emplvl", "annual_avg_estabs", "total_annual_wages", "avg_annual_pay"], ["cbsa_code", "naics_sector", "year"])
qcew_measures["suppressed_rows"] = int(qcew["disclosure_code"].fillna("").eq("N").sum())
qcew_measures

# CBP Feasibility

In [ ]:
cbp_url = "https://api.census.gov/data/2022/cbp?get=NAME,NAICS2017,NAICS2017_LABEL,ESTAB,EMP,PAYANN,ESTAB_F,EMP_F,PAYANN_F&for=metropolitan%20statistical%20area/micropolitan%20statistical%20area:26620&NAICS2017=*&LFO=001&EMPSZES=001"
cbp_response = requests.get(cbp_url, timeout=30)
{"status_code": cbp_response.status_code, "final_url": cbp_response.url, "requires_key": "missing_key" in cbp_response.url or "Missing Key" in cbp_response.text}

# ACS Feasibility

In [ ]:
acs_url = "https://api.census.gov/data/2023/acs/acs1/profile?get=NAME,DP05_0001E,DP03_0062E,DP02_0068PE,DP03_0002PE,DP03_0005PE&for=metropolitan%20statistical%20area/micropolitan%20statistical%20area:26620"
acs_response = requests.get(acs_url, timeout=30)
{"status_code": acs_response.status_code, "final_url": acs_response.url, "requires_key": "missing_key" in acs_response.url or "Missing Key" in acs_response.text}

# Merge Quality

In [ ]:
b_keep = ["cbsa_code", "pilot_label", "official_name", "naics_sector", "year", "estabs_entry", "estabs_entry_rate", "job_creation_births", "job_creation_rate_births"]
q_keep = ["cbsa_code", "naics_sector", "year", "annual_avg_emplvl", "annual_avg_estabs", "total_annual_wages", "avg_annual_pay", "employment_growth", "establishment_growth", "payroll_growth", "wage_growth"]
panel = bds[b_keep].merge(qcew[q_keep], on=["cbsa_code", "naics_sector", "year"], how="outer", indicator=True)
merge_report = pd.DataFrame([
    {"metric": "BDS pilot rows", "value": len(bds)},
    {"metric": "QCEW pilot rows", "value": len(qcew)},
    {"metric": "Outer merged rows", "value": len(panel)},
    {"metric": "Rows matched BDS and QCEW", "value": int(panel["_merge"].eq("both").sum())},
    {"metric": "BDS-only rows", "value": int(panel["_merge"].eq("left_only").sum())},
    {"metric": "QCEW-only rows", "value": int(panel["_merge"].eq("right_only").sum())},
    {"metric": "BDS-QCEW match percentage of merged rows", "value": pct(panel["_merge"].eq("both").sum(), len(panel))},
])
merge_report

# Entrepreneurship Measure Comparison

Establishment entry rate is the strongest pilot measure because it is normalized, interpretable, and mostly complete in the BDS pilot extract. Establishment entries and startup job creation births are useful backup measures, but raw counts are less comparable across MSAs.

# Industry Growth Variables

QCEW annual average employment, establishments, total annual wages, and average annual pay were retrieved for the pilot MSAs and sectors. Growth rates are nominal; payroll and wage growth are not labeled real because no inflation adjustment was applied.

# Expected Entrepreneurship Diagnostic

In [ ]:
model_df = panel.dropna(subset=["estabs_entry_rate", "employment_growth", "establishment_growth", "naics_sector", "cbsa_code"]).copy()
model = smf.ols("estabs_entry_rate ~ employment_growth + establishment_growth + C(naics_sector) + C(cbsa_code)", data=model_df).fit()
model_df["expected_estabs_entry_rate"] = model.predict(model_df)
model_df["entrepreneurial_residual"] = model_df["estabs_entry_rate"] - model_df["expected_estabs_entry_rate"]
{"nobs": int(model.nobs), "rsquared": round(float(model.rsquared), 3)}

# Gap Threshold Sensitivity

In [ ]:
resid = model_df["entrepreneurial_residual"]
thresholds = {
    "bottom_10_percent": resid <= resid.quantile(0.10),
    "bottom_20_percent": resid <= resid.quantile(0.20),
    "bottom_25_percent": resid <= resid.quantile(0.25),
    "below_minus_1_sd": resid < (-1 * resid.std()),
}
threshold_report = []
for name, mask in thresholds.items():
    threshold_report.append({
        "threshold": name,
        "gap_observations": int(mask.sum()),
        "percent_gap": pct(mask.sum(), len(model_df)),
        "gap_count_by_msa": model_df.loc[mask, "pilot_label"].value_counts().to_dict(),
        "top_gap_industries": model_df.loc[mask, "naics_sector"].value_counts().head(5).to_dict(),
    })
pd.DataFrame(threshold_report)

# Historical Window Assessment

A 2010-2023 full-project window appears more practical than 2005-2023 because it preserves useful history while reducing risks from geography, industry-classification, and ACS compatibility changes. A 2018-2023 pilot window worked for BDS and QCEW, while Census API-key access remains the main blocker for CBP and ACS retrieval.

# Feasibility Recommendation

GO WITH MODIFICATIONS: BDS and QCEW support the core pilot panel. CBP and ACS are conceptually compatible but require Census API-key access for live data retrieval and final merge testing.